# WaveGrip: CNN input images
## One trial, as each CNN sees it

Tristan Au

All the gesture-classification CNNs use the same ResNet-152 v1.5; only the input layer differs. This notebook takes **one training trial** and builds it into the input of each network exactly as its training notebook does, **including normalisation with training-set statistics**, then plots each input as the "image" the first convolution sees.

| CNN (notebook) | Input image (C × H × W) | Height | Width | Preprocessing → normalisation |
|---|---|---|---|---|
| Raw sEMG (`cnn_raw.ipynb`) | 1 × 12 × 10240 | 12 electrodes | 10240 samples = 5 s at 2048 Hz | counts → mV `(counts − baseline) / gain` → z-score per electrode |
| PSD (`cnn_psd.ipynb`) | 1 × 12 × 246 | 12 electrodes | 246 Welch bins, 10–500 Hz (2 Hz apart) | `log10(psd + 1e-12)` → z-score per (electrode, bin) |
| DWT (`cnn_dwt.ipynb`) | 1 × 12 × 5120 | 12 electrodes | db4 coefficients, bands joined end to end: A5 \| D5 \| D4 \| D3 \| D2 (320 + 320 + 640 + 1280 + 2560) | z-score per (band, electrode) → concatenate |
| DWT envelope (`cnn_dwt_envelope.ipynb`) | 5 × 12 × 320 | 12 electrodes | 320 frames of 15.6 ms; the 5 bands are image **channels** | frame energy → 78 ms moving average → `log10(· + 1e-12)` → z-score per (band, electrode) |
| DWT time-frequency (`cnn_dwt_tf_image.ipynb`) | 12 × 5 × 2560 | 5 DWT bands, A5 → D2 (the 12 electrodes are image **channels**) | 2560 columns of 1.95 ms; each band repeated onto this grid | z-score per (band, electrode) → repeat A5/D5 × 8, D4 × 4, D3 × 2, D2 × 1 (signed coefficients, no smoothing) |

**Faithful to training.** The preprocessing below is copied from each training notebook. Normalisation statistics are recomputed from the **training rows only**, exactly as training does; for the DWT run they are checked against the saved `*_normalization.npz`. The DWT, DWT-envelope and DWT time-frequency inputs go through a float16 round trip, because training kept them on the GPU in that format. The sample trial comes from the training split; **the test set is never read**.

**Colour.** Every panel uses the same diverging scale: red = above the training mean for that pixel's normalisation group, blue = below, grey ≈ mean, clipped at ±3 standard deviations. The meaning of "a pixel" differs per input: a voltage sample (Raw), a log-power bin (PSD), a wavelet coefficient (DWT and DWT time-frequency), or a frame's log band energy (envelope).

**Runtime.** The statistics need one pass over the Raw (3.8 GB) and DWT (3.8 GB) files: about 1–2 minutes locally. No GPU is needed. Figures are saved to `fine-tuning/input_images/`.

**Running on Google Colab.** Put the three datasets in `MyDrive/WaveGrip/ProcessedDatasets/` (same layout as the training notebooks) and run all cells; the configuration cell mounts Drive and copies them to local disk.

## 0. Configuration

In [ ]:
import shutil
import sys
import time
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap
from scipy.ndimage import uniform_filter1d
from sklearn.model_selection import train_test_split

DATA_FILES = {"raw": "raw_wrist/grabmyo_wrist_raw.h5", "psd": "psd_wrist/grabmyo_wrist_psd.h5",
              "dwt": "dwt_wrist/grabmyo_wrist_dwt.h5"}          # relative to ProcessedDatasets/; dwtenv uses the DWT file

# ---- Paths: Google Colab (Drive) or the local project folder --------------------------------------
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/WaveGrip")
    DATA_DIR = Path("/content/data")                            # Drive reads are slow: copy once per session
    for rel in DATA_FILES.values():
        src, dst = PROJECT_ROOT / "ProcessedDatasets" / rel, DATA_DIR / rel
        assert src.exists(), f"upload the dataset to {src}"
        if not dst.exists() or dst.stat().st_size != src.stat().st_size:
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(src, dst)
else:
    PROJECT_ROOT = Path.cwd().parent                            # run from fine-tuning/
    DATA_DIR = PROJECT_ROOT / "ProcessedDatasets"
H5 = {k: DATA_DIR / rel for k, rel in DATA_FILES.items()}
for p in H5.values():
    assert p.exists(), f"dataset not found at {p}"
RUNS_DIR = next((PROJECT_ROOT / "CNN" / d for d in ["Runs", "runs"] if (PROJECT_ROOT / "CNN" / d).is_dir()), None)
OUT_DIR = PROJECT_ROOT / "fine-tuning" / "input_images"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Settings copied from the training notebooks --------------------------------------------------
FS, N_SAMPLES = 2048, 10240
SPLIT, TEST_FRACTION, VAL_FRACTION, SEED = "random", 0.2, 0.2, 0
DWT_BANDS = ["A5", "D5", "D4", "D3", "D2"]                    # concatenation order (DWT) = channel order (envelope)
BAND_HZ = {"A5": "0–32 Hz", "D5": "32–64 Hz", "D4": "64–128 Hz", "D3": "128–256 Hz", "D2": "256–512 Hz"}
N_FRAMES, SMOOTH_FRAMES, LOG_EPS = 320, 5, 1e-12               # DWT-envelope settings (cnn_dwt_envelope.ipynb)
N_TIME = 2560                                                  # DWT time-frequency grid, 1.95 ms (cnn_dwt_tf_image.ipynb)
CHUNK = 512                                                    # rows per block when streaming the HDF5 files

# ---- Sample trial ---------------------------------------------------------------------------------
SAMPLE_GESTURE = 16                                            # 1-17; the first training trial of this gesture is shown
GESTURE_NAMES = ["Lateral Prehension", "Thumb Adduction", "Thumb and Little Finger Opposition",
                 "Thumb and Index Finger Opposition", "Thumb and Index Finger Extension",
                 "Thumb and Little Finger Extension", "Index and Middle Finger Extension",
                 "Little Finger Extension", "Index Finger Extension", "Thumb Finger Extension",
                 "Wrist Extension", "Wrist Flexion", "Forearm Supination", "Forearm Pronation",
                 "Hand Open", "Hand Close", "Rest"]

# ---- Plot style: one diverging scale for every input (blue <-> red, grey midpoint), clipped at +/-3 --------
Z_LIM = 3
Z_CMAP = LinearSegmentedColormap.from_list(
    "z_div", ["#0d366b", "#2a78d6", "#9ec5f4", "#f0efec", "#f5b3ad", "#e34948", "#8f1d1d"])
INK, INK_2, MUTED = "#0b0b0b", "#52514e", "#898781"
plt.rcParams.update({"font.family": "sans-serif", "axes.labelcolor": INK_2, "xtick.color": INK_2,
                     "ytick.color": INK_2, "axes.titlecolor": INK})
print(f"data: {DATA_DIR}\nruns: {RUNS_DIR}\nfigures -> {OUT_DIR}")

## 1. Split and sample trial

The same two-step split as every training notebook (train/test, then train/validation; stratified by participant × gesture, seeds 0 and 1). It is checked against any split a run folder saved. The sample is the first **training** trial of `SAMPLE_GESTURE`.

In [ ]:
with h5py.File(H5["dwt"]) as f:
    meta = {k: f[k][:].astype(np.int64) for k in ["session", "participant", "gesture", "trial"]}
for key in ["raw", "psd"]:                                     # a row index means the same trial in every file
    with h5py.File(H5[key]) as f:
        assert all(np.array_equal(meta[k], f[k][:]) for k in meta), f"{key} rows differ from dwt"
labels = meta["gesture"] - 1
N = len(labels)


def holdout(rows, fraction, seed):
    strata = meta["participant"][rows] * 100 + meta["gesture"][rows]
    return train_test_split(rows, test_size=fraction, stratify=strata, random_state=seed)


trainval_rows, _ = holdout(np.arange(N), TEST_FRACTION, SEED)
train_rows, _ = holdout(trainval_rows, VAL_FRACTION, SEED + 1)
TRAIN_ROWS = np.sort(train_rows)
TRAIN_MASK = np.zeros(N, bool)
TRAIN_MASK[TRAIN_ROWS] = True

if RUNS_DIR is not None:
    for path in sorted(RUNS_DIR.glob("*/*_split.npz")):
        assert np.array_equal(np.load(path)["train"], TRAIN_ROWS), f"{path.name}: different training rows"
        print(f"training rows match {path.name}")

ROW = int(TRAIN_ROWS[np.argmax(labels[TRAIN_ROWS] == SAMPLE_GESTURE - 1)])
SAMPLE = (f"training trial {ROW}: gesture {SAMPLE_GESTURE} ({GESTURE_NAMES[SAMPLE_GESTURE - 1]}), "
          f"participant {meta['participant'][ROW]}, session {meta['session'][ROW]}, trial {meta['trial'][ROW]}")
print(f"{len(TRAIN_ROWS)} training trials; sample = {SAMPLE}")

## 2. Build each CNN's input for the sample trial

Each block copies its training notebook's preprocessing. The DWT time-frequency image uses the same normalised coefficients as the DWT input, laid out differently (`cnn_dwt_tf_image.ipynb`). The statistics are accumulated over the training rows in float64, streaming the files in blocks of `CHUNK` rows (the same estimator as the training notebooks' `channel_mean_std`).

In [ ]:
class Stats:
    """Running mean/std over `axis`, accumulated in float64 (same estimator as training's channel_mean_std)."""

    def __init__(self, axis):
        self.axis, self.s, self.ss, self.n = axis, 0.0, 0.0, 0

    def add(self, x):
        x = x.astype(np.float64)
        self.s = self.s + x.sum(axis=self.axis)
        self.ss = self.ss + (x ** 2).sum(axis=self.axis)
        self.n += x.size // np.size(self.s)

    def result(self):
        mean = self.s / self.n
        std = np.sqrt(np.maximum(self.ss / self.n - mean ** 2, 0))
        return mean, np.where(std > 0, std, 1.0)


def band_envelope(coeffs, n_frames=N_FRAMES, smooth=SMOOTH_FRAMES, eps=LOG_EPS):
    """Copy of cnn_dwt_envelope.ipynb: DWT coefficients (..., L) -> log10 smoothed energy envelope (..., n_frames)."""
    per_frame = coeffs.shape[-1] // n_frames
    energy = (coeffs.astype(np.float64) ** 2).reshape(*coeffs.shape[:-1], n_frames, per_frame).mean(-1)
    if smooth > 1:
        energy = uniform_filter1d(energy, smooth, axis=-1, mode="nearest")
    return np.log10(energy + eps).astype(np.float32)


def f16(x):
    return x.astype(np.float16).astype(np.float32)             # training kept DWT / envelope inputs as float16 on the GPU


images, norm_notes = {}, {}
t0 = time.perf_counter()

# ---- Raw (cnn_raw.ipynb, RawInputs): mV, z-score per electrode ------------------------------------
with h5py.File(H5["raw"]) as f:
    gain, baseline = f["gain"][:], f["baseline"][:].astype(np.float64)
    st = Stats(axis=(0, 2))
    for a in range(0, N, CHUNK):
        m = TRAIN_MASK[a:a + CHUNK]
        if m.any():
            st.add((f["emg"][a:a + CHUNK][m] - baseline[a:a + CHUNK][m][..., None]) / gain[a:a + CHUNK][m][..., None])
    mean, std = st.result()
    mv = (f["emg"][ROW] - baseline[ROW][:, None]) / gain[ROW][:, None]
images["raw"] = ((mv - mean[:, None]) / std[:, None]).astype(np.float32)[None]           # (1, 12, 10240)
print(f"raw    done ({time.perf_counter() - t0:.0f} s)")

# ---- PSD (cnn_psd.ipynb, load_psd): log10, z-score per (electrode, bin) ---------------------------
with h5py.File(H5["psd"]) as f:
    x = np.log10(f["psd"][:].astype(np.float64) + 1e-12)
    FREQS = f["freqs"][:]
mean, std = x[TRAIN_ROWS].mean(axis=0), x[TRAIN_ROWS].std(axis=0)
std = np.where(std > 0, std, 1.0)
images["psd"] = ((x[ROW] - mean) / std).astype(np.float32)[None]                         # (1, 12, 246)
del x
print(f"psd    done ({time.perf_counter() - t0:.0f} s)")

# ---- DWT (cnn_dwt.ipynb, load_dwt) and DWT envelope (cnn_dwt_envelope.ipynb, load_envelopes): one pass --
dwt_stats = {b: Stats(axis=(0, 2)) for b in DWT_BANDS}
env_stats = {b: Stats(axis=(0, 2)) for b in DWT_BANDS}
with h5py.File(H5["dwt"]) as f:
    g = f["dwt"]
    for a in range(0, N, CHUNK):
        m = TRAIN_MASK[a:a + CHUNK]
        if not m.any():
            continue
        for b in DWT_BANDS:
            c = g[b][a:a + CHUNK][m]
            dwt_stats[b].add(c)
            env_stats[b].add(band_envelope(c))
    sample = {b: g[b][ROW] for b in DWT_BANDS}
dwt_parts, env_parts, dwt_norm = [], [], {}
for b in DWT_BANDS:
    mean, std = dwt_stats[b].result()
    dwt_norm[f"{b}_mean"], dwt_norm[f"{b}_std"] = mean, std
    dwt_parts.append((sample[b] - mean[:, None]) / std[:, None])
    mean, std = env_stats[b].result()
    env_parts.append((band_envelope(sample[b]) - mean[:, None]) / std[:, None])
images["dwt"] = f16(np.concatenate(dwt_parts, axis=-1))[None]                            # (1, 12, 5120)
images["dwtenv"] = f16(np.stack(env_parts))                                              # (5, 12, 320)
images["dwttf"] = f16(np.stack([np.repeat(p, N_TIME // p.shape[-1], axis=-1) for p in dwt_parts], axis=1))  # (12, 5, 2560)
print(f"dwt    done ({time.perf_counter() - t0:.0f} s)")

saved = RUNS_DIR / "dwt_random_resnet152" / "dwt_random_resnet152_normalization.npz" if RUNS_DIR else None
if saved is not None and saved.exists():                       # recomputed statistics == the ones training saved
    ref = np.load(saved)
    for k, v in dwt_norm.items():
        np.testing.assert_allclose(v, ref[k], rtol=1e-4)
    print(f"DWT statistics match {saved.name}")

EXPECTED = {"raw": (1, 12, 10240), "psd": (1, 12, 246), "dwt": (1, 12, 5120), "dwtenv": (5, 12, 320), "dwttf": (12, 5, 2560)}
for k, img in images.items():
    assert img.shape == EXPECTED[k] and np.isfinite(img).all(), (k, img.shape)
    print(f"{k:>7}: {img.shape}, mean {img.mean():6.2f}, std {img.std():5.2f}, "
          f"{100 * (np.abs(img) > Z_LIM).mean():4.1f} % of pixels beyond ±{Z_LIM}")

## 3. Plot helpers

Every panel is drawn with `imshow` at its true pixel count. Raw (10240 columns) and DWT (5120 columns) have more columns than the figure has screen pixels (about 1650 across). With `interpolation="nearest"`, matplotlib then draws only the nearest input column for each screen pixel: about 1 in 6 columns for Raw and 1 in 3 for DWT, skipping the rest. The full-width panels are therefore an overview, not every value the CNN sees. Those two inputs also get a **zoom panel** with at least one screen pixel per input column, so every value in that window is drawn.

In [ ]:
ELECTRODE_TICKS = ([0, 5, 6, 11], ["W1", "W6", "W7", "W12"])


def panel(ax, img, extent=None, xlabel=None, ylabel="Electrode"):
    im = ax.imshow(img, aspect="auto", cmap=Z_CMAP, vmin=-Z_LIM, vmax=Z_LIM, interpolation="nearest",
                   extent=extent if extent is None else (*extent, 11.5, -0.5))
    ax.set_yticks(*ELECTRODE_TICKS)
    ax.set_ylabel(ylabel)
    if xlabel:
        ax.set_xlabel(xlabel)
    ax.spines[:].set_visible(False)
    return im


def colorbar(fig, im, axes):
    cb = fig.colorbar(im, ax=axes, fraction=0.025, pad=0.01)
    cb.set_label(f"normalised input value (z-score, clipped at ±{Z_LIM})", color=INK_2)
    cb.outline.set_visible(False)


def title(fig, name, shape):
    fig.suptitle(f"{name} CNN input ({' × '.join(map(str, shape))}): {SAMPLE}", x=0.01, ha="left", color=INK)


def save(fig, key):
    fig.savefig(OUT_DIR / f"input_image_{key}.png", dpi=150)
    plt.show()


DWT_OFFSETS = np.cumsum([0, 320, 320, 640, 1280, 2560])


def mark_dwt_bands(ax, lo=0, hi=5120):
    for b, a, z in zip(DWT_BANDS, DWT_OFFSETS[:-1], DWT_OFFSETS[1:]):
        if z <= lo or a >= hi:
            continue
        if lo < a:
            ax.axvline(a, color=INK, lw=1.2)
        ax.text((max(a, lo) + min(z, hi)) / 2, -0.7, f"{b}\n{BAND_HZ[b]}", ha="center", va="bottom",
                fontsize=8, linespacing=1.1, color=INK)

## 4. Raw sEMG CNN: 1 × 12 × 10240

Each column is one sample of the band-passed signal (0.49 ms). The full-width panel draws only about 1 in 6 samples, so it looks like random stripes: it shows roughly where amplitude is large, but not the waveform. The zoom shows 0.25 s, where each pixel is one sample: the signal swings between red and blue every few samples, often on many electrodes at once.

In [ ]:
ZOOM_S = (0.10, 0.35)                                          # seconds shown in the zoom panel
img = images["raw"][0]
lo, hi = (int(t * FS) for t in ZOOM_S)
fig, (a1, a2) = plt.subplots(2, 1, figsize=(13, 6.5), constrained_layout=True)
im = panel(a1, img, extent=(0, N_SAMPLES / FS), xlabel="Time in trial (s), 10240 samples at 2048 Hz")
a1.axvspan(*ZOOM_S, fill=False, ec=INK, lw=1)
a1.set_title("Full trial", loc="left", fontsize=10)
panel(a2, img[:, lo:hi], extent=(lo / FS, hi / FS), xlabel=f"Time in trial (s): zoom on {hi - lo} samples")
a2.set_title(f"Zoom, {ZOOM_S[0]:.2f}–{ZOOM_S[1]:.2f} s (one pixel per sample)", loc="left", fontsize=10)
colorbar(fig, im, [a1, a2])
title(fig, "Raw sEMG", images["raw"].shape)
save(fig, "raw")

## 5. PSD CNN: 1 × 12 × 246

Each column is one 2 Hz Welch bin between 10 and 500 Hz; each pixel is that electrode's log power at that frequency over the whole 5 s trial, relative to the training average for the same (electrode, bin). There is no time axis.

In [ ]:
img = images["psd"][0]
df = FREQS[1] - FREQS[0]
fig, ax = plt.subplots(figsize=(13, 3.6), constrained_layout=True)
im = panel(ax, img, extent=(FREQS[0] - df / 2, FREQS[-1] + df / 2),
           xlabel=f"Frequency (Hz), {len(FREQS)} Welch bins {FREQS[0]:g}–{FREQS[-1]:g} Hz")
colorbar(fig, im, ax)
title(fig, "PSD", images["psd"].shape)
save(fig, "psd")

## 6. DWT CNN: 1 × 12 × 5120

The five bands are joined along the width (black lines mark the joins). Each band runs over the **whole** 5 s trial, so the time axis restarts at every join, and its time step shrinks band by band (15.6 ms in A5/D5 to 1.95 ms in D2). The zoom covers the D5 | D4 | D3 joins: detail coefficients often flip sign from one column to the next (neighbouring columns correlate at r ≈ −0.25 over the training trials), and the kernels at each join see two different bands side by side.

In [ ]:
ZOOM_COLS = (560, 1400)                                        # columns shown in the zoom panel (D5 | D4 | D3)
img = images["dwt"][0]
fig, (a1, a2) = plt.subplots(2, 1, figsize=(13, 6.8), constrained_layout=True)
im = panel(a1, img, xlabel="Column (coefficient index); each band spans the whole 5 s trial")
mark_dwt_bands(a1)
a1.axvspan(*ZOOM_COLS, fill=False, ec=INK, lw=1, ls="--")
a1.set_title("All 5120 columns", loc="left", fontsize=10, pad=26)
panel(a2, img[:, slice(*ZOOM_COLS)], extent=ZOOM_COLS, xlabel=f"Column: zoom on {ZOOM_COLS[1] - ZOOM_COLS[0]} coefficients")
mark_dwt_bands(a2, *ZOOM_COLS)
a2.set_title(f"Zoom, columns {ZOOM_COLS[0]}–{ZOOM_COLS[1]}", loc="left", fontsize=10, pad=26)
colorbar(fig, im, [a1, a2])
title(fig, "DWT", images["dwt"].shape)
save(fig, "dwt")

## 7. DWT-envelope CNN: 5 × 12 × 320

The five bands are the image's **channels** (like R, G, B), each drawn as its own panel. All channels share one time axis of 320 frames (15.6 ms), so a burst appears in the same columns of every band, and the first 7 × 7 convolution mixes all five bands at the same moment.

In [ ]:
img = images["dwtenv"]
fig, axes = plt.subplots(len(DWT_BANDS), 1, figsize=(13, 9.5), sharex=True, constrained_layout=True)
for k, (ax, b) in enumerate(zip(axes, DWT_BANDS)):
    im = panel(ax, img[k], extent=(0, N_SAMPLES / FS), ylabel=f"ch {k}: {b}\n{BAND_HZ[b]}")
axes[-1].set_xlabel(f"Time in trial (s), {N_FRAMES} frames of {1000 * N_SAMPLES / FS / N_FRAMES:.1f} ms")
colorbar(fig, im, axes)
title(fig, "DWT envelope", images["dwtenv"].shape)
save(fig, "dwtenv")

## 8. DWT time-frequency CNN: 12 × 5 × 2560

The same DWT coefficients as Section 6, laid out as one **frequency × time image per electrode**: the 12 electrodes are the image's channels (one panel each), the rows are the 5 bands (low frequency at the bottom), and the columns are one shared 0–5 s time grid of 1.95 ms. Each band's coefficients are repeated onto that grid, so a column holds the coefficient of every band that covers the same moment. Values stay signed, so the red/blue texture of the DWT input remains, but time is now continuous and lined up across bands.

The full panels have more columns (2560) than screen pixels, so they draw about every 2nd column. The zoom shows electrode W6 over 0.25 s with every column drawn: A5/D5 change every 8 columns, D4 every 4, D3 every 2 and D2 every column. A5 runs about 31 ms behind the other bands (the db4 filter delay at the deepest level).

In [ ]:
ZOOM_ELECTRODE = 5                                             # W6; time window ZOOM_S from Section 4
img = images["dwttf"]                                          # (12, 5, N_TIME)


def tf_panel(ax, a, t0=0.0, t1=N_SAMPLES / FS, labels=DWT_BANDS):
    im = ax.imshow(a, aspect="auto", origin="lower", cmap=Z_CMAP, vmin=-Z_LIM, vmax=Z_LIM, interpolation="nearest",
                   extent=(t0, t1, -0.5, len(DWT_BANDS) - 0.5))
    ax.set_yticks(range(len(DWT_BANDS)), labels)
    ax.tick_params(labelsize=8)
    ax.spines[:].set_visible(False)
    return im


fig, axes = plt.subplots(6, 2, figsize=(14, 11), sharex=True, constrained_layout=True)
for e, ax in enumerate(axes.T.flat):                           # W1-W6 left column, W7-W12 right
    im = tf_panel(ax, img[e])
    ax.set_title(f"ch {e}: W{e + 1}", loc="left", fontsize=9)
for ax in axes[-1]:
    ax.set_xlabel(f"Time in trial (s), {N_TIME} columns of {1000 * N_SAMPLES / FS / N_TIME:.2f} ms")
colorbar(fig, im, axes)
title(fig, "DWT time-frequency", img.shape)
save(fig, "dwttf")

lo, hi = (int(round(t * N_TIME * FS / N_SAMPLES)) for t in ZOOM_S)
fig, ax = plt.subplots(figsize=(13, 3.4), constrained_layout=True)
im = tf_panel(ax, img[ZOOM_ELECTRODE][:, lo:hi], lo * N_SAMPLES / FS / N_TIME, hi * N_SAMPLES / FS / N_TIME,
              labels=[f"{b} ({BAND_HZ[b]})" for b in DWT_BANDS])
ax.set_xlabel(f"Time in trial (s): {hi - lo} columns, every column drawn")
ax.set_title(f"Zoom: channel {ZOOM_ELECTRODE} (W{ZOOM_ELECTRODE + 1}), {ZOOM_S[0]:.2f}–{ZOOM_S[1]:.2f} s", loc="left", fontsize=10)
colorbar(fig, im, ax)
save(fig, "dwttf_zoom")

## 9. All inputs side by side

The same trial in every input, on one colour scale. The DWT envelope has 5 channels, so it takes five rows. The DWT time-frequency image has 12 channels; one of them (electrode W6, the strongest in this trial) is shown, with the 5 bands as rows.

In [ ]:
OVERVIEW_ELECTRODE = 5                                         # W6: the time-frequency channel shown here
T_EXT = (0, N_SAMPLES / FS)
rows = [("Raw sEMG\n1 × 12 × 10240", images["raw"][0], T_EXT, "Time (s)"),
        ("PSD\n1 × 12 × 246", images["psd"][0], (FREQS[0] - df / 2, FREQS[-1] + df / 2), "Frequency (Hz)"),
        ("DWT\n1 × 12 × 5120", images["dwt"][0], (0, 5120), "Column (A5 | D5 | D4 | D3 | D2)")]
rows += [(f"DWT envelope\nch {k}: {b}", images["dwtenv"][k], T_EXT,
          "Time (s)" if k == len(DWT_BANDS) - 1 else None) for k, b in enumerate(DWT_BANDS)]
fig, axes = plt.subplots(len(rows) + 1, 1, figsize=(13, 17),
                         gridspec_kw={"height_ratios": [1.6, 1.6, 1.6] + [1] * len(DWT_BANDS) + [1.3]},
                         constrained_layout=True)
for ax, (name, img, ext, xl) in zip(axes, rows):
    im = panel(ax, img, extent=ext, xlabel=xl, ylabel=name)
    ax.yaxis.label.set(rotation=0, ha="right", va="center", color=INK)
mark_dwt_bands(axes[2])
ax_tf = axes[-1]                                               # DWT time-frequency: one of its 12 channels
tf_panel(ax_tf, images["dwttf"][OVERVIEW_ELECTRODE])
ax_tf.set_ylabel(f"DWT time-frequency\n12 × 5 × 2560\nch {OVERVIEW_ELECTRODE}: W{OVERVIEW_ELECTRODE + 1}",
                 rotation=0, ha="right", va="center", color=INK)
ax_tf.set_xlabel("Time (s)")
env_axes = axes[3:3 + len(DWT_BANDS)]
for ax in list(env_axes[1:]) + [ax_tf]:
    ax.sharex(env_axes[0])
for ax in env_axes[:-1]:
    ax.tick_params(labelbottom=False)
colorbar(fig, im, axes)
fig.suptitle(f"Input images of the CNNs: {SAMPLE}", x=0.01, ha="left", color=INK)
save(fig, "all")

## 10. Summary

In [ ]:
summary = pd.DataFrame([
    ("Raw sEMG", "cnn_raw.ipynb", images["raw"].shape, "12 electrodes", "time: 10240 samples (0.49 ms)", "per electrode"),
    ("PSD", "cnn_psd.ipynb", images["psd"].shape, "12 electrodes", f"frequency: {len(FREQS)} bins (2 Hz)", "per (electrode, bin)"),
    ("DWT", "cnn_dwt.ipynb", images["dwt"].shape, "12 electrodes", "5 bands joined; time restarts per band", "per (band, electrode)"),
    ("DWT envelope", "cnn_dwt_envelope.ipynb", images["dwtenv"].shape, "12 electrodes", "time: 320 frames (15.6 ms); bands = channels", "per (band, electrode)"),
    ("DWT time-frequency", "cnn_dwt_tf_image.ipynb", images["dwttf"].shape, "5 bands (A5 → D2)", "time: 2560 columns (1.95 ms); electrodes = channels", "per (band, electrode)"),
], columns=["CNN", "Notebook", "Input (C, H, W)", "Height", "Width", "z-score groups"]).set_index("CNN")
display(summary)
print("figures:", *sorted(p.name for p in OUT_DIR.glob("input_image_*.png")), sep="\n  ")